In [ ]:
# Система прогнозирования продаж для «Прилавка»

## Описание задачи

Клиент — компания «Прилавок», новая сеть дарксторов в России, специализирующаяся на быстрой доставке продуктов и товаров повседневного спроса. Компания работает с 2023 года и управляет более чем 45 дарксторами.

**Бизнес-проблема**

Эффективность дарксторов критически зависит от точности управления запасами. Текущая система планирования закупок работает на субъективных оценках аналитиков, что приводит к разным проблемам:
- К дефициту товаров в пиковые периоды (недополучение 3–5% выручки).
- К избыточным запасам и списаниям продуктов с коротким сроком годности (2–3% от выручки).
- К неэффективному использованию времени аналитиков (15–20 часов в неделю на ручное планирование).

**Цель проекта**

Разработать автоматизированную систему прогнозирования недельных продаж для каждого отдела во всех дарксторах с использованием машинного обучения (модель CatBoost). Система будет работать в режиме пакетного внедрения с недельным циклом: модель будет переобучаться каждое воскресенье вечером, а к понедельнику утром прогнозы будут готовы для передачи в систему планирования закупок.

## Описание данных

Для работы доступны исторические данные о продажах за февраль 2023 — октябрь 2025, организованные в виде нескольких таблиц в PostgreSQL:

1. Таблица `sales` — исторические данные о продажах:
   - `Store` — номер даркстора (1–45).
   - `Dept` — номер отдела внутри даркстора (1–99).
   - `Date` — неделя (дата начала недели).
   - `Weekly_Sales` — недельные продажи (руб., целевая переменная).
   - `IsHoliday` — признак праздничной недели.

2. Таблица `stores` — информация о дарксторах:
   - `Store` — номер даркстора.
   - `Type` — тип даркстора (A, B, C).
   - `Size` — размер даркстора в условных единицах.

3. Таблица `features` — внешние факторы:
   - `Store`, `Date` — идентификаторы.
   - `Temperature` — средняя температура за неделю.
   - `Fuel_Price` — средняя цена бензина.
   - `MarkDown1-5` — факторы рекламных акций и скидок.
   - `CPI` — индекс потребительских цен.
   - `Unemployment` — уровень безработицы.

4. Таблица `plan` — данные для инференса (ноябрь 2025 — июль 2026), содержит те же столбцы, что и `sales`, кроме `Weekly_Sales`.

## План работы

Проект состоит из двух частей. В этом ноутбуке вы выполните первую:

1. Первичный анализ и очистка данных.
2. Предобработка и создание признаков.
3. Мониторинг стабильности признаков (PSI).
4. Обучение модели CatBoost и оценка качества.



# Здесь все необходимые импорты

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import psycopg2

import warnings
warnings.filterwarnings("ignore")

Фиксируем `random_state` для воспроизводимости результатов.

RANDOM_STATE = 42

### Шаг 1. Первичный анализ данных

import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    os.getenv('DB_USER'),
    os.getenv('DB_PASSWORD'),
    os.getenv('DB_HOST'),
    os.getenv('DB_PORT'),
    os.getenv('DB_NAME')
)

engine = create_engine(connection_string)

query_sales = """
SELECT *
FROM public.sales
"""

sales_df = pd.read_sql_query(query_sales, con=engine)
sales_df.head()

query_stores = """
SELECT *
FROM public.stores
"""

stores_df = pd.read_sql_query(query_stores, con=engine)
stores_df.head()

query_features = """
SELECT *
FROM public.features
"""

features_df = pd.read_sql_query(query_features, con=engine)
features_df.head()

sales_df.info()

stores_df.info()

features_df.info()

print("Sales - Дубликаты:", sales_df.duplicated().sum())
print("Sales - Пропуски:\n", sales_df.isna().sum())
sales_df.drop_duplicates(inplace=True)
print("\nStores - Дубликаты:", stores_df.duplicated().sum())
print("Stores - Пропуски:\n", stores_df.isna().sum())
stores_df.drop_duplicates(inplace=True)
print("\nFeatures - Дубликаты:", features_df.duplicated().sum())
print("Features - Пропуски до обработки:\n", features_df.isna().sum())
features_df.drop_duplicates(inplace=True)
factor_cols = ['factor2', 'factor3', 'factor4', 'factor5']
for col in factor_cols:
    mean_value = features_df[col].mean()
    features_df[col].fillna(mean_value, inplace=True)

print("\nFeatures - Пропуски после обработки:\n", features_df.isna().sum())

sales_df['date'] = pd.to_datetime(sales_df['date'])
weekly_sales = sales_df.groupby('date')['weekly_sales'].sum().reset_index()
sales_df['month'] = sales_df['date'].dt.to_period('M')
monthly_sales = sales_df.groupby('month')['weekly_sales'].sum().reset_index()
monthly_sales['month'] = monthly_sales['month'].dt.to_timestamp()
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 10))

plt.subplot(2, 1, 1)
sns.lineplot(data=weekly_sales, x='date', y='weekly_sales', color='royalblue', linewidth=2)
plt.title('Распределение недельных продаж (Выявление сезонности и всплесков)', fontsize=14)
plt.xlabel('Дата (Недели)', fontsize=12)
plt.ylabel('Продажи (руб.)', fontsize=12)

plt.subplot(2, 1, 2)
sns.barplot(data=monthly_sales, x='month', y='weekly_sales', color='teal')
plt.title('Распределение месячных продаж (Общий тренд)', fontsize=14)
plt.xlabel('Дата (Месяцы)', fontsize=12)
plt.ylabel('Продажи (руб.)', fontsize=12)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Приводим дату в features_df к datetime для корректного merge
features_df['date'] = pd.to_datetime(features_df['date'])

# Шаг 1: Объединяем продажи с информацией о дарксторах
df = pd.merge(sales_df, stores_df, on='store', how='left')

# Шаг 2: Добавляем внешние факторы (features)
# Объединяем по store, dept и date
df = pd.merge(df, features_df, on=['store', 'dept', 'date'], how='left')

# Проверяем финальную структуру
print("Формат итогового датасета:", df.shape)
print("\nПервые 5 строк объединенной таблицы:")
df.head()

### Матрица корреляций

from phik import phik_matrix

matrix = df.phik_matrix(interval_cols=df.select_dtypes(include=['float','int']).columns)
plt.figure(figsize=(20,20))
sns.heatmap(data=matrix, fmt='.2f',annot=True)
plt.title('Тепловая карта матрицы корреляций')
plt.show()

**обработка аномалий в продажах.**

negative_sales = sales_df[sales_df['weekly_sales'] <= 0]
print(f"Количество строк с продажами <= 0: {len(negative_sales)}")
print(f"Процент от общего объема данных: {len(negative_sales) / len(sales_df) * 100:.4f}%")
print("\nПример таких записей:")
print(negative_sales.head())

sales_clean_df = sales_df[sales_df['weekly_sales'] > 0].copy()
print(f"Размер датасета после удаления отрицательных продаж: {sales_clean_df.shape[0]}")

quantile_threshold = sales_clean_df['weekly_sales'].quantile(0.9999)
print(f"Порог для топ-0.01% продаж: {quantile_threshold:,.2f} руб.")
high_sales_anomalies = sales_clean_df[sales_clean_df['weekly_sales'] > quantile_threshold].copy()
print(f"Количество записей в топе: {len(high_sales_anomalies)}")

high_sales_rich = high_sales_anomalies.merge(stores_df, on='store', how='left')
high_sales_rich['date'] = pd.to_datetime(high_sales_rich['date'])
high_sales_rich['month'] = high_sales_rich['date'].dt.month
high_sales_rich['year'] = high_sales_rich['date'].dt.year

print("Распределение пиковых продаж по месяцам:")
print(high_sales_rich['month'].value_counts().sort_index())

print("\nСвязь с праздничными неделями (is_holiday):")
print(high_sales_rich['is_holiday'].value_counts(normalize=True) * 100)

print("Какого типа дарксторы чаще генерят сверхвыручку?")
print(high_sales_rich['type'].value_counts())

print("\nТоп-5 отделов (dept) с аномально высокими продажами:")
print(high_sales_rich['dept'].value_counts().head(5))

### Шаг 2. Предобработка признаков и feature engineering

def create_temporal_features(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['year'] = df['date'].dt.year
    return df

df = create_temporal_features(df)

assert 'month' in df.columns, 'Месяц не создан!'
assert 'quarter' in df.columns, 'Квартал не создан!'
assert 'year' in df.columns, 'Год не создан!'

def create_avg_sales_feature(df):
    df = df.copy()
    df = df.sort_values(by=['store', 'dept', 'date']).reset_index(drop=True)    
    df['avg_sales_before'] = (
        df.groupby(['store', 'dept'])['weekly_sales']
        .apply(lambda x: x.expanding().mean().shift(1))
        .reset_index(level=[0, 1], drop=True)
    )
    return df

df = create_avg_sales_feature(df)

assert 'avg_sales_before' in df.columns, 'Признак средних продаж не создан!'

def create_lag_features(df):
    df = df.copy()
    df = df.sort_values(by=['store', 'dept', 'date']).reset_index(drop=True)
    df['sales_1week_ago'] = df.groupby(['store', 'dept'])['weekly_sales'].shift(1)
    df['sales_2week_ago'] = df.groupby(['store', 'dept'])['weekly_sales'].shift(2)
    df['sales_4week_ago'] = df.groupby(['store', 'dept'])['weekly_sales'].shift(4)
    return df

df = create_lag_features(df)

assert 'sales_1week_ago' in df.columns, 'Признак sales_1week_ago не создан!'
assert 'sales_2week_ago' in df.columns, 'Признак sales_2week_ago не создан!'
assert 'sales_4week_ago' in df.columns, 'Признак sales_4week_ago не создан!'

def create_rolling_features(df):
    df = df.copy()
    df = df.sort_values(by=['store', 'dept', 'date']).reset_index(drop=True)
    
    df['sales_shifted_for_rolling'] = df.groupby(['store', 'dept'])['weekly_sales'].shift(1)
    
    df['mean_sales_2week'] = (
        df.groupby(['store', 'dept'])['sales_shifted_for_rolling']
        .transform(lambda x: x.rolling(window=2, min_periods=2).mean())
    )
    
    df['mean_sales_4week'] = (
        df.groupby(['store', 'dept'])['sales_shifted_for_rolling']
        .transform(lambda x: x.rolling(window=4, min_periods=4).mean())
    )
    
    df.drop(columns=['sales_shifted_for_rolling'], inplace=True)
    return df

df = create_rolling_features(df)

assert 'mean_sales_2week' in df.columns, 'Признак mean_sales_2week не создан!'
assert 'mean_sales_4week' in df.columns, 'Признак mean_sales_4week не создан!'

**удаляем строки с пропусками.**

lag_cols = ['avg_sales_before', 'sales_1week_ago', 'sales_2week_ago', 'sales_4week_ago', 'mean_sales_2week', 'mean_sales_4week']
print("Пропуски в новых признаках до очистки:")
print(df[lag_cols].isna().sum())

print(f"\nРазмер датасета ДО удаления пропусков: {df.shape}")
df.dropna(subset=lag_cols, inplace=True)
print(f"Размер датасета ПОСЛЕ удаления пропусков: {df.shape}")

df.reset_index(drop=True, inplace=True)

### Шаг 3. Мониторинг стабильности признаков

df = df.sort_values('date').reset_index(drop=True)

train_df = df[df['date'] < '2025-09-01'].copy()
val_df = df[(df['date'] >= '2025-09-01') & (df['date'] <= '2025-10-31')].copy()

print(f"Обучение (Train): {train_df['date'].min().strftime('%Y-%m-%d')} — {train_df['date'].max().strftime('%Y-%m-%d')} | Строк: {len(train_df)}")
print(f"Валидация (Validation): {val_df['date'].min().strftime('%Y-%m-%d')} — {val_df['date'].max().strftime('%Y-%m-%d')} | Строк: {len(val_df)}")

### Расчет PSI

def calculate_psi(actual, expected, num_buckets=10):
    actual = actual[~np.isnan(actual)]
    expected = expected[~np.isnan(expected)]
    
    percentiles = np.linspace(0, 100, num_buckets + 1)
    buckets = np.percentile(actual, percentiles)
    
    buckets[0] -= 1e-5
    buckets[-1] += 1e-5
    
    actual_counts, _ = np.histogram(actual, bins=buckets)
    expected_counts, _ = np.histogram(expected, bins=buckets)
    
    actual_pcts = actual_counts / len(actual)
    expected_pcts = expected_counts / len(expected)
    
    actual_pcts = np.where(actual_pcts == 0, 1e-4, actual_pcts)
    expected_pcts = np.where(expected_pcts == 0, 1e-4, expected_pcts)
    
    psi_value = np.sum((actual_pcts - expected_pcts) * np.log(actual_pcts / expected_pcts))
    return psi_value

all_columns = train_df.columns.tolist()

excluded_cols = ['weekly_sales', 'date', 'year']
features_to_check = [col for col in all_columns if col not in excluded_cols]

psi_results = {}

for col in features_to_check:
    if train_df[col].dtype == 'object' or train_df[col].dtype == 'bool':
        actual_values = train_df[col].astype('category').cat.codes.values
        expected_values = val_df[col].astype('category').cat.codes.values
    else:
        actual_values = train_df[col].values
        expected_values = val_df[col].values
        
    psi_val = calculate_psi(actual_values, expected_values, num_buckets=10)
    psi_results[col] = psi_val

psi_df = pd.DataFrame(list(psi_results.items()), columns=['Признак (Feature)', 'PSI']).sort_values(by='PSI', ascending=False)

def interpret_psi(psi):
    if psi < 0.1:
        return 'PSI < 0.1 — стабилен, можно использовать'
    elif psi < 0.2:
        return '0.1 ≤ PSI < 0.2 — умеренный дрейф, требуется внимание'
    else:
        return 'PSI ≥ 0.2 — значительный дрейф, признак нестабилен'

psi_df['Статус'] = psi_df['PSI'].apply(interpret_psi)

print(f"Всего проверено признаков: {len(psi_df)}")
print(psi_df.to_string(index=False))

bad_features = [
    'month',
    'quarter',
    'fuel_price',
    'factor5',
    'cpi',
    'factor3',
    'factor1',
    'factor4',
    'temperature',
    'factor2',
    'unemployment',
    'is_holiday'
]

train_df_cleaned = train_df.drop(columns=bad_features, errors='ignore')

val_df_cleaned = val_df.drop(columns=bad_features, errors='ignore')

print("Осталось признаков в Train:", train_df_cleaned.shape[1] - 2) # минусуем таргет и дату
print("Оставшиеся колонки:", train_df_cleaned.columns.tolist())

### Шаг 4. Обучение модели CatBoost

from catboost import CatBoostRegressor

X_train = train_df_cleaned.drop(columns=['weekly_sales', 'date'], errors='ignore')
y_train = train_df_cleaned['weekly_sales']

X_val = val_df_cleaned.drop(columns=['weekly_sales', 'date'], errors='ignore')
y_val = val_df_cleaned['weekly_sales']

cat_features = [col for col in ['store', 'dept', 'type'] if col in X_train.columns]
for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_seed=RANDOM_STATE,
    verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    early_stopping_rounds=50
)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

y_pred = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

print("--- Метрики качества на валидационной выборке ---")
print(f"MAE:  {mae:,.2f} руб.")
print(f"RMSE: {rmse:,.2f} руб.")
print(f"R²:   {r2:.4f}")
print("-" * 50)

## Важность признаков.


importance = model.get_feature_importance()
feature_importance_df = pd.DataFrame({
    'Признак': X_train.columns,
    'Важность (%)': importance
}).sort_values(by='Важность (%)', ascending=False)

print("\n--- Важность признаков для модели ---")
print(feature_importance_df.to_string(index=False))

### Сохранение модели в S3

import io
import os
import pickle
import boto3
from dotenv import load_dotenv

load_dotenv()
cb = model
s3_client = boto3.client(
    's3',
    endpoint_url='https://storage.yandexcloud.net',
    aws_access_key_id=os.getenv('S3_ACCESS_KEY'),
    aws_secret_access_key=os.getenv('S3_SECRET_KEY'),
    region_name='ru-central1'
)

filebuffer = io.BytesIO()
pickle.dump(cb, filebuffer)
filebuffer.seek(0)
bucket_name = os.getenv('S3_BUCKET')

s3_client.upload_fileobj(
    Fileobj=filebuffer,
    Bucket=bucket_name,
    Key='catboost_model.pkl'  
)

print(f"Модель успешно загружена в {bucket_name}/catboost_model.pkl")